In [ ]:
%pip install pandas

In [ ]:
import pandas as pd
from pathlib import Path

# Work whether Jupyter starts in the project root or Instagram folder.
instagram_dir = Path.cwd() if (Path.cwd() / 'Raw Data').is_dir() else Path.cwd() / 'Instagram'
raw_data_dir = instagram_dir / 'Raw Data'
output_path = instagram_dir / 'instagram_cleanded.csv'
files = {
    2019: raw_data_dir / '2026-07-10_klscm2019.json',
    2023: raw_data_dir / '2026-07-10_klscm2023.json',
    2024: raw_data_dir / '2026-07-10_klscm2024.json',
    2025: raw_data_dir / '2026-07-10_klscm2025.json',
}
missing_files = [str(path) for path in files.values() if not path.exists()]
assert not missing_files, f'Missing input files: {missing_files}'
yearly_stats = []

In [ ]:
def clean_year(raw_df, year, old_schema):
    author_ids = (
        raw_df['author'].map(lambda value: value.get('id') if isinstance(value, dict) else None)
        if old_schema else raw_df['ownerId']
    )
    timestamp_column = 'taken_at_timestamp' if old_schema else 'timestamp'
    cleaned = pd.DataFrame({
        'author_id': author_ids,
        'post_url': raw_df['url'],
        'caption': raw_df['caption'].astype('string').str.strip(),
        'hashtag': f'klscm{year}',
        'timestamp': raw_df[timestamp_column],
    })
    blank_mask = cleaned['caption'].isna() | cleaned['caption'].eq('')
    blank_count = int(blank_mask.sum())
    cleaned = cleaned.loc[~blank_mask].copy()
    before_dedup = len(cleaned)
    cleaned = cleaned.drop_duplicates(['author_id', 'caption']).reset_index(drop=True)
    duplicate_count = before_dedup - len(cleaned)
    yearly_stats.append({
        'year': year, 'raw_total': len(raw_df),
        'blank_captions_removed': blank_count,
        'duplicates_removed': duplicate_count,
        'total_removed': len(raw_df) - len(cleaned),
        'cleaned_total': len(cleaned),
    })
    return cleaned

In [ ]:
df_2019_raw = pd.read_json(files[2019])
raw_total_2019 = len(df_2019_raw)
print(f'2019 raw total: {raw_total_2019:,}')
df_2019_raw.head(1)

In [ ]:
df_2019 = clean_year(df_2019_raw, 2019, old_schema=True)
print(f'2019 cleaned total: {len(df_2019):,}')
df_2019.head()

In [ ]:
df_2023_raw = pd.read_json(files[2023])
raw_total_2023 = len(df_2023_raw)
print(f'2023 raw total: {raw_total_2023:,}')
df_2023_raw.head(1)

In [ ]:
df_2023 = clean_year(df_2023_raw, 2023, old_schema=True)
print(f'2023 cleaned total: {len(df_2023):,}')
df_2023.head()

In [ ]:
df_2024_raw = pd.read_json(files[2024])
raw_total_2024 = len(df_2024_raw)
print(f'2024 raw total: {raw_total_2024:,}')
df_2024_raw.head(1)

In [ ]:
df_2024 = clean_year(df_2024_raw, 2024, old_schema=False)
print(f'2024 cleaned total: {len(df_2024):,}')
df_2024.head()

In [ ]:
df_2025_raw = pd.read_json(files[2025])
raw_total_2025 = len(df_2025_raw)
print(f'2025 raw total: {raw_total_2025:,}')
df_2025_raw.head(1)

In [ ]:
df_2025 = clean_year(df_2025_raw, 2025, old_schema=False)
print(f'2025 cleaned total: {len(df_2025):,}')
df_2025.head()

In [ ]:
yearly_summary = pd.DataFrame(yearly_stats)
assert (yearly_summary['raw_total'] - yearly_summary['total_removed'] == yearly_summary['cleaned_total']).all()
yearly_summary

In [ ]:
final_columns = ['author_id', 'post_url', 'caption', 'hashtag', 'timestamp']
combined_df = pd.concat([df_2019, df_2023, df_2024, df_2025], ignore_index=True)
combined_before_deduplication = len(combined_df)
combined_df['author_id'] = combined_df['author_id'].astype('string')
combined_df['timestamp'] = pd.to_datetime(combined_df['timestamp'], utc=True, errors='coerce')
combined_df = combined_df[final_columns]
assert combined_before_deduplication == int(yearly_summary['cleaned_total'].sum())
print(f'Combined before final deduplication: {combined_before_deduplication:,}')

In [ ]:
total_raw = int(yearly_summary['raw_total'].sum())
instagram_cleaned = combined_df.drop_duplicates(['author_id', 'caption']).copy()
cross_year_duplicates_removed = combined_before_deduplication - len(instagram_cleaned)
instagram_cleaned = instagram_cleaned.sort_values('timestamp', na_position='last').reset_index(drop=True)
final_combined_total = len(instagram_cleaned)
print(f'Cross-year duplicates removed: {cross_year_duplicates_removed:,}')
print(f'Final combined total: {final_combined_total:,}')

In [ ]:
final_totals = pd.DataFrame([{
    'total_raw': total_raw,
    'combined_before_final_deduplication': combined_before_deduplication,
    'cross_year_duplicates_removed': cross_year_duplicates_removed,
    'final_combined_total': final_combined_total,
}])
assert list(instagram_cleaned.columns) == final_columns
assert not instagram_cleaned['caption'].isna().any()
assert not instagram_cleaned['caption'].eq('').any()
assert not instagram_cleaned.duplicated(['author_id', 'caption']).any()
assert set(instagram_cleaned['hashtag']) == {'klscm2019', 'klscm2023', 'klscm2024', 'klscm2025'}
assert not instagram_cleaned['timestamp'].isna().any(), 'Some timestamps could not be parsed.'
display(instagram_cleaned.head())
display(instagram_cleaned['hashtag'].value_counts().rename_axis('hashtag').reset_index(name='final_count'))
display(instagram_cleaned.isna().sum().rename('null_count').to_frame())
print('Remaining duplicate author-caption pairs:', int(instagram_cleaned.duplicated(['author_id', 'caption']).sum()))
display(final_totals)

In [ ]:
instagram_cleaned.to_csv(output_path, index=False, encoding='utf-8-sig')
export_check = pd.read_csv(output_path, dtype={'author_id': 'string'})
assert list(export_check.columns) == final_columns
assert len(export_check) == final_combined_total
print(f'Exported {final_combined_total:,} rows to: {output_path.resolve()}')